Answer the problem in the context of topics covered in SDS. You are **not allowed** to seek or receive help from anyone to solve the problem. To keep the problem at reasonable difficulty, you are also **not allowed** to use LLMs such as ChatGPT, Gemini, Claude and Llama. You can **only use built-in libraries** of Apache Spark. For example, you cannot use Spark packages.

# Problem 1 [25 pts]

In this problem, you will design and implement a function that will return the most similar users of a user based on their commonly listened songs.

## Problem 1a [10 pts]

Discuss the step-by-step process of how you will transform a set of song IDs into a locally-sensitive hash. When performing approximate nearest neighbor search, a Jaccard similarity of at least 80% should result in a 90% chance of collision and at most 10% chance of collision if their Jaccard similarity is at most 50%. Be as detailed as possible. Explain the reasoning for each step. Make sure all parameter values are explicitly stated (not just the equation) and justified. 

You should include and specify the following but you may add more information as necessary:

* Distance function that you will use
* Locality-sensitive function that you will use
* Signature size
* Number of bands

## Solution to Problem 1a

We need to transform each user's listened songs into a representation that supports approximate nearest-neighbor search. Since each user is represented by a **set of songs**, the natural similarity measure is **Jaccard similarity**.

### 1. Represent each user as a set

For each user \(u\), define:

$$
S_u = \{\text{song IDs listened to by user } u\}
$$

Repeated plays do not matter for this problem because the prompt asks about **commonly listened songs**, and the required similarity threshold is based on **Jaccard similarity**, which is set-based.

For example, if a user played the same song 20 times, that song still appears once in the user's set.

---

### 2. Distance function

The similarity between two users \(u\) and \(v\) is:

$$
J(S_u,S_v)=\frac{|S_u \cap S_v|}{|S_u \cup S_v|}
$$

The corresponding distance is:

$$
d_J(S_u,S_v)=1-J(S_u,S_v)
$$

I use Jaccard similarity/distance because:

1. the data is naturally a set of song IDs;
2. the ordering of songs does not matter;
3. repeated plays should not dominate the result;
4. MinHash is specifically designed for Jaccard similarity.

---

### 3. Locality-sensitive function

The locality-sensitive function is **MinHash**.

For each hash function \(h_i\), the MinHash value of a user's song set is:

$$
\operatorname{minhash}_i(S_u)=\min_{x \in S_u} h_i(x)
$$

The important MinHash property is:

$$
P[\operatorname{minhash}_i(S_u)=\operatorname{minhash}_i(S_v)] = J(S_u,S_v)
$$

So if two users have high Jaccard similarity, their MinHash signatures are likely to agree in many positions.

---

### 4. Banding scheme

The problem requires:

- users with Jaccard similarity at least \(0.80\) should collide with probability at least \(0.90\);
- users with Jaccard similarity at most \(0.50\) should collide with probability at most \(0.10\).

For MinHash banding, if we use:

- \(b\) bands;
- \(r\) rows per band;
- signature size \(m = br\);

then the probability of at least one band collision for similarity \(s\) is:

$$
P(\text{collision}) = 1-(1-s^r)^b
$$

I choose:

$$
b=10
$$

$$
r=7
$$

Therefore, the signature size is:

$$
m = br = 10 \times 7 = 70
$$

Check the high-similarity requirement:

$$
P(\text{collision}\mid s=0.80)
=
1-(1-0.80^7)^{10}
\approx 0.905
$$

This is at least \(0.90\), so it satisfies the high-similarity requirement.

Check the low-similarity requirement:

$$
P(\text{collision}\mid s=0.50)
=
1-(1-0.50^7)^{10}
\approx 0.075
$$

This is at most \(0.10\), so it satisfies the low-similarity requirement.

---

### 5. Final parameter values

The final LSH design is:

| Component | Choice |
|---|---:|
| Similarity | Jaccard similarity |
| Distance | Jaccard distance |
| LSH family | MinHash |
| Signature size | 70 |
| Number of bands | 10 |
| Rows per band | 7 |
| Collision probability at \(s=0.80\) | about 0.905 |
| Collision probability at \(s=0.50\) | about 0.075 |

This means a pair of users with at least 80% Jaccard similarity has about a 90.5% chance of becoming a candidate pair, while a pair with at most 50% Jaccard similarity has only about a 7.5% chance of becoming a candidate pair.

# Problem 1b [15 pts]

The file `/mnt/data/public/userid_songid_count.txt` is a tab-separated file consisting of the userid, songid, and the number of times the song was played by the user. Using Apache Spark and this file, create a function that will take in the `userid` and return a list of most similar users (`userid`) according to the criteria set in Problem 1a.

## Solution to Problem 1b

The implementation below follows the design from Problem 1a:

1. Read the tab-separated file.
2. Convert user-song-play-count rows into unique user-song pairs.
3. Build 70 MinHash values per user.
4. Split each 70-value signature into 10 bands of 7 rows.
5. For a target user, find candidate users that collide in at least one band.
6. Compute exact Jaccard similarity only for those candidates.
7. Return the most similar users.

This avoids the earlier `CountVectorizer.fit()` approach, which may crash Spark on large datasets because it tries to build a very large vocabulary of songs.

In [ ]:
# ============================================================
# Spark setup
# ============================================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Problem1_Manual_MinHash_LSH")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

In [ ]:
# ============================================================
# LSH parameters from Problem 1a
# ============================================================

NUM_BANDS = 10
ROWS_PER_BAND = 7
SIGNATURE_SIZE = NUM_BANDS * ROWS_PER_BAND

# A large prime used only as a modulus for keeping hash values bounded.
# Spark's xxhash64 already gives a good distributed hash, but pmod keeps
# the values nonnegative and easier to compare.
HASH_MODULUS = 4_294_967_291

print("Number of bands:", NUM_BANDS)
print("Rows per band:", ROWS_PER_BAND)
print("Signature size:", SIGNATURE_SIZE)

In [ ]:
# ============================================================
# 1. Load the file
# ============================================================

data_path = "/mnt/data/public/userid_songid_count.txt"

df = (
    spark.read
    .option("sep", "\t")
    .option("header", "false")
    .csv(data_path)
    .toDF("userid", "songid", "play_count")
)

df = df.withColumn("play_count", F.col("play_count").cast("int"))

df.show(5, truncate=False)
df.printSchema()

In [ ]:
# ============================================================
# 2. Create unique user-song pairs
# ============================================================
# Jaccard similarity is set-based. Therefore, repeated plays are ignored
# when constructing the set of songs for each user.

user_song_pairs = (
    df
    .select("userid", "songid")
    .dropna()
    .dropDuplicates(["userid", "songid"])
)

user_song_pairs.cache()

print("Unique user-song pairs:", user_song_pairs.count())
user_song_pairs.show(5, truncate=False)

In [ ]:
# ============================================================
# 3. Optional safety filter for local notebook testing
# ============================================================
# On very large datasets, building all 70 MinHash values for all user-song
# pairs can still be expensive. For exam/testing notebooks, it is useful
# to start with a manageable subset, then increase the limit.
#
# IMPORTANT:
# The sample target user must be explicitly included in the working set.
# Otherwise, the last example cell may fail even if the code is correct.

USE_LIMIT = True
MAX_USERS_FOR_TESTING = 5000

TARGET_USER = "b80344d063b5ccb3212f76538f3d9e43d87dca9e"

if USE_LIMIT:
    # Select a manageable number of users.
    selected_users = (
        user_song_pairs
        .select("userid")
        .distinct()
        .limit(MAX_USERS_FOR_TESTING)
    )

    # Always include the target user used in the example call.
    target_user_df = (
        user_song_pairs
        .filter(F.col("userid") == TARGET_USER)
        .select("userid")
        .distinct()
    )

    selected_users = selected_users.unionByName(target_user_df).distinct()

    working_pairs = user_song_pairs.join(selected_users, on="userid", how="inner")
else:
    working_pairs = user_song_pairs

working_pairs.cache()

print("Working user-song pairs:", working_pairs.count())
print("Working users:", working_pairs.select("userid").distinct().count())

# Verify that the target user is present.
target_present = working_pairs.filter(F.col("userid") == TARGET_USER).limit(1).count()
print("Target user included:", bool(target_present))

In [ ]:
# ============================================================
# 4. Build song sets per user
# ============================================================

user_sets = (
    working_pairs
    .groupBy("userid")
    .agg(F.collect_set("songid").alias("songs"))
)

user_sets.cache()

print("Users with song sets:", user_sets.count())
user_sets.show(5, truncate=False)

In [ ]:
# ============================================================
# 5. Compute the 70-value MinHash signature for each user
# ============================================================
# We simulate 70 independent hash functions by hashing the pair:
#
#     hash_id || songid
#
# For each user and each hash_id, the MinHash value is the minimum hash
# value over all songs in the user's set.

hash_ids = spark.range(SIGNATURE_SIZE).withColumnRenamed("id", "hash_id")

hashed_pairs = (
    working_pairs
    .crossJoin(hash_ids)
    .withColumn(
        "hash_value",
        F.pmod(
            F.xxhash64(
                F.concat_ws("::", F.col("hash_id").cast("string"), F.col("songid"))
            ),
            F.lit(HASH_MODULUS)
        )
    )
)

minhash_signature_long = (
    hashed_pairs
    .groupBy("userid", "hash_id")
    .agg(F.min("hash_value").alias("minhash_value"))
)

minhash_signature_long.cache()

minhash_signature_long.show(10, truncate=False)

In [ ]:
# ============================================================
# 6. Convert signatures into band keys
# ============================================================
# We use:
# - 10 bands
# - 7 rows per band
#
# Two users become candidate matches if at least one full band is identical.

signature_with_bands = (
    minhash_signature_long
    .withColumn("band_id", (F.col("hash_id") / F.lit(ROWS_PER_BAND)).cast("int"))
    .withColumn("row_in_band", (F.col("hash_id") % F.lit(ROWS_PER_BAND)).cast("int"))
)

band_signatures = (
    signature_with_bands
    .groupBy("userid", "band_id")
    .agg(
        F.sort_array(
            F.collect_list(
                F.struct(
                    F.col("row_in_band").alias("row_in_band"),
                    F.col("minhash_value").alias("minhash_value")
                )
            )
        ).alias("band_entries")
    )
    .withColumn(
        "band_key",
        F.sha2(
            F.concat_ws(
                "|",
                F.expr("transform(band_entries, x -> cast(x.minhash_value as string))")
            ),
            256
        )
    )
    .select("userid", "band_id", "band_key")
)

band_signatures.cache()

band_signatures.show(10, truncate=False)

In [ ]:
# ============================================================
# 7. Define the required function
# ============================================================
# The function returns a Spark DataFrame with the most similar users.
#
# Steps:
# 1. Get the target user's band signatures.
# 2. Join on identical band_id and band_key to get LSH candidates.
# 3. Compute exact Jaccard similarity only among candidates.
# 4. If there are no LSH candidates, fall back to exact scoring over
#    the working sample so the function still returns a useful answer.
# 5. Return the top N users.

def most_similar_users(target_userid, top_n=5):
    target_exists = (
        user_sets
        .filter(F.col("userid") == target_userid)
        .limit(1)
        .count()
    )

    if target_exists == 0:
        raise ValueError(
            f"Target user '{target_userid}' was not found in the working dataset. "
            "If USE_LIMIT=True, make sure the target user is explicitly included "
            "in the selected_users DataFrame."
        )

    target_bands = (
        band_signatures
        .filter(F.col("userid") == target_userid)
        .select(
            F.col("band_id"),
            F.col("band_key")
        )
    )

    candidate_users = (
        target_bands
        .join(band_signatures, on=["band_id", "band_key"], how="inner")
        .select("userid")
        .where(F.col("userid") != target_userid)
        .distinct()
    )

    candidate_count = candidate_users.count()
    print("LSH candidate users found:", candidate_count)

    if candidate_count == 0:
        print(
            "No users collided with the target under the strict 10-band x 7-row "
            "LSH rule. Falling back to exact Jaccard scoring over the working sample."
        )

        candidate_users = (
            user_sets
            .select("userid")
            .where(F.col("userid") != target_userid)
        )

    target_song_set = (
        user_sets
        .filter(F.col("userid") == target_userid)
        .select(F.col("songs").alias("target_songs"))
    )

    candidate_song_sets = (
        candidate_users
        .join(user_sets, on="userid", how="inner")
    )

    scored_candidates = (
        candidate_song_sets
        .crossJoin(target_song_set)
        .withColumn("intersection_size", F.size(F.array_intersect("songs", "target_songs")))
        .withColumn("union_size", F.size(F.array_union("songs", "target_songs")))
        .withColumn(
            "JaccardSimilarity",
            F.col("intersection_size") / F.col("union_size")
        )
        .withColumn(
            "JaccardDistance",
            F.lit(1.0) - F.col("JaccardSimilarity")
        )
        .select("userid", "JaccardSimilarity", "JaccardDistance")
        .orderBy(F.col("JaccardSimilarity").desc(), F.col("userid").asc())
        .limit(top_n)
    )

    return scored_candidates

In [ ]:
# ============================================================
# 8. Example call
# ============================================================
# Use the requested sample target user if present. If it is not in the
# loaded/sample dataset, automatically choose a valid fallback user.

requested_target_user = TARGET_USER

target_exists = (
    user_sets
    .filter(F.col("userid") == requested_target_user)
    .limit(1)
    .count()
)

if target_exists > 0:
    target_user = requested_target_user
    print("Using requested target user:", target_user)
else:
    fallback_row = (
        user_sets
        .select("userid", F.size("songs").alias("num_songs"))
        .orderBy(F.col("num_songs").desc(), F.col("userid").asc())
        .first()
    )
    if fallback_row is None:
        raise ValueError("No users are available in user_sets.")
    target_user = fallback_row["userid"]
    print("Requested target user was not found in the working dataset.")
    print("Using fallback target user:", target_user)
    print("Fallback user's number of songs:", fallback_row["num_songs"])

similar = most_similar_users(target_user, top_n=5)
similar.show(truncate=False)

## Fix applied

The previous run failed in the final cell because the target user was not included in the sampled working dataset.

The safety-filter cell now explicitly includes:

```python
TARGET_USER = "b80344d063b5ccb3212f76538f3d9e43d87dca9e"
```

It also unions that user into the sampled users before building `working_pairs`.

The function also includes a fallback: if no users collide under the strict LSH banding rule, it computes exact Jaccard similarity over the working sample so the final cell still returns a useful result.

## Interpretation

The result is ordered by highest exact Jaccard similarity among users that passed the MinHash LSH candidate step.

The columns mean:

| Column | Meaning |
|---|---|
| `userid` | similar user ID |
| `JaccardSimilarity` | fraction of shared songs over total distinct songs |
| `JaccardDistance` | \(1 - \text{JaccardSimilarity}\) |

Higher `JaccardSimilarity` means the user is more similar to the target user.

This is an approximate nearest-neighbor method because LSH is used to avoid comparing the target user against every other user exactly. Exact Jaccard similarity is computed only after the candidate filtering step.

## Note on Spark stability

The earlier implementation used:

```python
CountVectorizer(inputCol="songs", outputCol="features").fit(user_sets)
```

That can crash Spark if the number of unique songs is large, because Spark has to build a large vocabulary and sparse feature space.

The revised implementation avoids that bottleneck. It computes MinHash signatures directly from `(userid, songid)` rows using Spark SQL functions.

In [ ]:
# ============================================================
# Helper: choose a valid target user
# ============================================================
# The original prompt/sample uses this user ID:
REQUESTED_TARGET_USER = "b80344d063b5ccb3212f76538f3d9e43d87dca9e"

def choose_valid_target_user(requested_userid=REQUESTED_TARGET_USER):
    """
    Return requested_userid if it exists in user_sets.
    Otherwise, return a valid user from the working dataset.

    This prevents the final example from failing when:
    1. the requested user is not in the actual file, or
    2. the requested user was excluded by sampling.
    """
    requested_exists = (
        user_sets
        .filter(F.col("userid") == requested_userid)
        .limit(1)
        .count()
    )

    if requested_exists > 0:
        print("Using requested target user:", requested_userid)
        return requested_userid

    fallback_row = (
        user_sets
        .select("userid", F.size("songs").alias("num_songs"))
        .orderBy(F.col("num_songs").desc(), F.col("userid").asc())
        .first()
    )

    if fallback_row is None:
        raise ValueError("No users are available in user_sets.")

    fallback_user = fallback_row["userid"]

    print("Requested target user was not found in the working dataset.")
    print("Using fallback target user:", fallback_user)
    print("Fallback user's number of songs:", fallback_row["num_songs"])

    return fallback_user

In [ ]:
# ============================================================
# 8. Example call
# ============================================================
# This cell no longer fails if the sample target user is unavailable.
# It uses the requested target if present; otherwise, it chooses a valid user.

target_user = choose_valid_target_user()

similar = most_similar_users(target_user, top_n=5)
similar.show(truncate=False)

## Final-cell robustness fix

The original sample target user:

```python
"b80344d063b5ccb3212f76538f3d9e43d87dca9e"
```

may not exist in the working dataset, especially when `USE_LIMIT=True`.

The notebook now checks whether that user exists. If not, it automatically selects a valid user from `user_sets`, preferring a user with many songs so the similarity result is more meaningful.